# Análise de Dados de População (2010-2022)

In [ ]:
import pandas as pd
import os

# Definir o caminho do arquivo
arquivo = r'H:\Py3etapa\At4\CD2022_Populacao_2010_Compatibilizada_20231222.xlsx'

# Ler o arquivo Excel
df = pd.read_excel(arquivo)

# Exibir as primeiras linhas
print("Dados carregados com sucesso!")
print(f"Dimensões: {df.shape}")
print("\nPrimeiras linhas:")
print(df.head())
print("\nColunas:")
print(df.columns.tolist())
print("\nTipos de dados:")
print(df.dtypes)

## 1. Agregação por Estado

In [ ]:
# Agregar por estado - identificando a coluna de estado
# Supondo que existe uma coluna de estado
print("Colunas disponíveis:")
print(df.columns.tolist())

# Exibir alguns dados para entender a estrutura
print("\nSample dos dados:")
print(df.head(20))

In [ ]:
# Agregação por Estado - somando as colunas numéricas
# Identificar qual coluna é de estado e quais são numéricas

# Selecionar colunas numéricas (exceto índices)
colunas_numericas = df.select_dtypes(include=['number']).columns.tolist()
print(f"Colunas numéricas encontradas: {colunas_numericas}")

# Assumindo que a primeira coluna não numérica ou uma coluna específica é estado
# Vamos agrupar - precisamos identificar a coluna de estado
print("\nColuna de Estado (primeiras valores únicos):")
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype == 'string':
        print(f"Coluna '{col}': {df[col].nunique()} valores únicos")
        print(f"  Exemplos: {df[col].unique()[:5]}")

In [ ]:
# Agregar por Estado
# Adaptar conforme a estrutura real dos dados

# Encontrar a coluna de estado (geralmente é uma coluna de texto com poucos valores únicos)
coluna_estado = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() < 50:  # Estados são limitados
        coluna_estado = col
        break

if coluna_estado:
    print(f"Coluna de Estado identificada: {coluna_estado}")
    
    # Agregar por estado
    df_estado = df.groupby(coluna_estado)[colunas_numericas].sum().reset_index()
    
    # Se houver colunas para 2010 e 2022, calcular a diferença
    print("\nColunas numéricas para criar diferença:")
    print(colunas_numericas)
    
    # Visualizar resultado
    print("\nDados agregados por estado:")
    print(df_estado.head(10))
else:
    print("Coluna de estado não identificada automaticamente.")
    print("Colunas disponíveis:")
    print(df.columns.tolist())

In [ ]:
# Calcular crescimento populacional por estado (2022 - 2010)
# Adaptar os nomes das colunas conforme necessário

# Verificar quais colunas correspondem a 2010 e 2022
print("Colunas disponíveis:")
print(df.columns.tolist())

# Procurar colunas que contenham '2010' e '2022'
col_2010 = [col for col in df.columns if '2010' in str(col)]
col_2022 = [col for col in df.columns if '2022' in str(col)]

print(f"\nColunas 2010: {col_2010}")
print(f"Colunas 2022: {col_2022}")

In [ ]:
# Se encontrou as colunas, calcular crescimento
if col_2010 and col_2022 and coluna_estado:
    col_2010 = col_2010[0]
    col_2022 = col_2022[0]
    
    # Calcular diferença
    df_estado['Crescimento'] = df_estado[col_2022] - df_estado[col_2010]
    
    # Ordenar por crescimento (decrescente)
    df_estado_ordenado = df_estado.sort_values('Crescimento', ascending=False).reset_index(drop=True)
    
    print("Estados ordenados por crescimento populacional (2022 - 2010):")
    print(df_estado_ordenado)
    
    # Salvar em CSV
    caminho_csv_estado = r'H:\Py3etapa\At4\populacao_por_estado.csv'
    df_estado_ordenado.to_csv(caminho_csv_estado, sep=';', index=False, encoding='utf-8')
    print(f"\n✓ Arquivo salvo: {caminho_csv_estado}")
else:
    print("Não foi possível identificar as colunas de 2010 e 2022.")

## 2. Agregação por Município

In [ ]:
# Identificar coluna de município
print("Análise das colunas para encontrar município:")
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"Coluna '{col}': {df[col].nunique()} valores únicos")
        
# Se houver coluna de município diferente de estado
coluna_municipio = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() > 50:  # Municípios são muitos
        coluna_municipio = col
        break

if coluna_municipio:
    print(f"\nColuna de Município identificada: {coluna_municipio}")
    print(f"Total de municípios: {df[coluna_municipio].nunique()}")
else:
    print("\nColuna de município não encontrada ou os dados já estão por município.")

In [ ]:
# Agregar por município se existir coluna de município
if coluna_municipio and coluna_municipio != coluna_estado:
    # Agregar por município
    df_municipio = df.groupby([coluna_estado, coluna_municipio])[colunas_numericas].sum().reset_index()
    
    # Calcular crescimento
    df_municipio['Crescimento'] = df_municipio[col_2022] - df_municipio[col_2010]
    
    # Ordenar por crescimento
    df_municipio_ordenado = df_municipio.sort_values('Crescimento', ascending=False).reset_index(drop=True)
    
    print("Municípios ordenados por crescimento populacional (2022 - 2010):")
    print(df_municipio_ordenado.head(20))
    
    # Salvar em CSV
    caminho_csv_municipio = r'H:\Py3etapa\At4\populacao_por_municipio.csv'
    df_municipio_ordenado.to_csv(caminho_csv_municipio, sep=';', index=False, encoding='utf-8')
    print(f"\n✓ Arquivo salvo: {caminho_csv_municipio}")
    print(f"Total de registros: {len(df_municipio_ordenado)}")
else:
    print("Não foi possível agregar por município (coluna não encontrada ou dados já agregados).")
    
    # Alternativa: se os dados já são por município, ordenar diretamente
    if col_2010 and col_2022:
        df['Crescimento'] = df[col_2022] - df[col_2010]
        df_sorted = df.sort_values('Crescimento', ascending=False).reset_index(drop=True)
        
        print("\nMunicípios ordenados por crescimento (dados originais):")
        print(df_sorted[[coluna_estado, coluna_municipio or 'Nome', col_2010, col_2022, 'Crescimento']].head(20))
        
        # Salvar
        caminho_csv_municipio = r'H:\Py3etapa\At4\populacao_por_municipio.csv'
        df_sorted.to_csv(caminho_csv_municipio, sep=';', index=False, encoding='utf-8')
        print(f"\n✓ Arquivo salvo: {caminho_csv_municipio}")

## Resumo Final

In [ ]:
print("=" * 60)
print("RESUMO DA ANÁLISE")
print("=" * 60)
print(f"✓ Dados originais carregados: {df.shape[0]} linhas")
print(f"✓ Tabela por estado criada e salva: populacao_por_estado.csv")
print(f"✓ Tabela por município criada e salva: populacao_por_municipio.csv")
print(f"\nArquivos salvos em: H:\\Py3etapa\\At4\\")
print("=" * 60)